In [ ]:
import os
import pandas as pd
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from sklearn.model_selection import train_test_split
from PIL import Image
import torch.nn.functional as F

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image

In [ ]:
class_labels={"Potato___Early_blight":0,"Potato___Late_blight":1,"Potato___healthy":2}

In [ ]:
class CustomeData(Dataset):
    def __init__(self, root_dir,split, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.split=split
        self.image_paths=[]
        self.labels=[]
        for class_name, label in class_labels.items():
            class_images = glob.glob(f"{root_dir}/{split}/{class_name}/*.JPG")  # Find all images
            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))  # Assign labels



    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
         # Use the csv to get paths
        label =self.labels[idx]
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)



        # Replace mask values with remapped values


        return image,label

In [ ]:
root_dir=os.path.join(path,"PlantVillage")
print(root_dir)

In [ ]:
import torchvision.transforms as transforms

In [ ]:
transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),

])
test_dataset=CustomeData(root_dir,split="test",transform=transforms)
train_dataset=CustomeData(root_dir,split="train",transform=transforms)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

In [ ]:
# Write your code here
class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
         super(CNNModel, self).__init__()
         self.conv1=nn.Conv2d(3, 96, kernel_size=11, stride=2, padding=5)

         self.maxpool=nn.MaxPool2d(kernel_size=3, stride=2)               # Output: 96 x 7 x 7
         self.conv2= nn.Conv2d(96, 256, kernel_size=5, padding=2)         # Output: 256 x 7 x 7
         self.conv3=nn.Conv2d(256, 384, kernel_size=3, padding=1)
               # Output: 256 x 3 x 3
         self.relu=nn.ReLU()
         self.conv4=nn.Conv2d(384, 384, kernel_size=3, padding=1)       # Output: 384 x 3 x 3

         self.conv5=nn.Conv2d(384, 384, kernel_size=3, padding=1)
               # Output: 384 x 3 x 3

                                # Output: 256 x 2 x 2
         self.softmax = nn.Softmax(dim=1)
         self.fc1= nn.Linear(3456, 4096)
         self.fc2 = nn.Linear(4096, num_classes)



    def forward(self, x):
        x = self.relu(self.conv1(x))
        x=self.relu(self.conv2(x))
        x=self.relu(self.conv3(x))
        x=self.maxpool(self.relu(self.conv4(x)))
        x=self.maxpool(self.relu(self.conv5(x)))
        x = torch.flatten(x, start_dim=1)
        x=self.relu(self.fc1(x))
        x=self.fc2(x)


        return x


In [ ]:
# Write your code here
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode, you will understand why later
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device),labels.to(device) # Move data to GPU if available


        outputs = model(images)
        # Forward pass
        loss = criterion(outputs, labels)  # Compute loss


        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation (compute gradients)
        optimizer.step()  # Update model parameters
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total_loss += loss.item()
        total += labels.size(0)
        # Collect the loss

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)

    return accuracy,avg_loss # Return average loss


In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode, you will understand why later
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient calculation
        for images, labels in tqdm(dataloader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # (Optional) Compute accuracy
            predictions = outputs.argmax(dim=1)  # Get class with highest probability
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return  avg_loss,accuracy

In [ ]:
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 3 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

In [ ]:
# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses ,label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
